In [5]:
import geopandas as gpd
import fiona

gpkg_path = r"D:\M2_MoSEF\Acclim\DataCollection\data\adminexpress-cog-simpl-000-2025.gpkg"  # <-- change this

# 1) List layers in the GeoPackage
layers = fiona.listlayers(gpkg_path)
print(" GPKG opened.")
print("Layers found:")
for i, lyr in enumerate(layers, 1):
    print(f"  {i:02d}. {lyr}")

# 2) Try to auto-pick a likely "communes" layer
# (Admin Express often contains 'COMMUNE', 'COMMUNE_CARTO', etc.)
candidates = [lyr for lyr in layers if "COMM" in lyr.upper() or "COM" in lyr.upper()]
print("\nLikely commune-ish layers:", candidates)

# pick the first candidate, else the first layer
layer_to_test = candidates[0] if candidates else layers[0]
print("\nTesting layer:", layer_to_test)

# 3) Load that layer and print quick diagnostics
gdf = gpd.read_file(gpkg_path, layer=layer_to_test)

print("\n--- Layer diagnostics ---")
print("Rows:", len(gdf))
print("CRS:", gdf.crs)
print("Geometry types:", gdf.geom_type.value_counts().to_dict())
print("Columns:", list(gdf.columns))

print("\n--- Head (first 5 rows) ---")
print(gdf.head(5))


 GPKG opened.
Layers found:
  01. commune_int
  02. commune
  03. departement
  04. region
  05. cheflieu
  06. epci
  07. departement_int
  08. region_int
  09. epci_int

Likely commune-ish layers: ['commune_int', 'commune']

Testing layer: commune_int

--- Layer diagnostics ---
Rows: 102177
CRS: EPSG:4326
Geometry types: {'LineString': 101846, 'MultiLineString': 331}
Columns: ['geometry']

--- Head (first 5 rows) ---
                                            geometry
0    LINESTRING (5.96855 47.90474, 5.96749 47.90476)
1    LINESTRING (5.96749 47.90476, 5.96152 47.91054)
2  LINESTRING (5.96152 47.91054, 5.96514 47.91313...
3  LINESTRING (5.97616 47.93422, 5.98861 47.93445...
4  LINESTRING (6.00841 47.92277, 6.01649 47.92003...


In [6]:
import geopandas as gpd

gpkg_path = r"D:\M2_MoSEF\Acclim\DataCollection\data\adminexpress-cog-simpl-000-2025.gpkg"  # <-- change this
layer = "commune"

gdf = gpd.read_file(gpkg_path, layer=layer)

print("Layer:", layer)
print("Rows:", len(gdf))
print("CRS:", gdf.crs)
print("Geometry types:", gdf.geom_type.value_counts().to_dict())
print("Columns:", list(gdf.columns))

# Show likely ID/name columns if present
likely = [c for c in gdf.columns if any(k in c.upper() for k in ["INSEE", "CODE", "NOM", "NAME", "LIB"])]
print("Likely commune id/name columns:", likely)

print(gdf.head(3))


Layer: commune
Rows: 34877
CRS: EPSG:4326
Geometry types: {'Polygon': 34760, 'MultiPolygon': 117}
Columns: ['population', 'surf_adminexpress_geo_ha', 'id', 'siren_epci', 'insee_reg', 'insee_dep', 'insee_arr', 'insee_can', 'statut', 'insee_com', 'surf_adminexpress_prj_ha', 'nom', 'geometry']
Likely commune id/name columns: ['insee_reg', 'insee_dep', 'insee_arr', 'insee_can', 'insee_com', 'nom']
   population  surf_adminexpress_geo_ha                        id siren_epci  \
0       594.0                    912.94  COMMUNE_0000000009742144  200036150   
1       907.0                    405.78  COMMUNE_0000000009742542  248900383   
2     13944.0                   1060.73  COMMUNE_0000000009733020  200068047   

  insee_reg insee_dep insee_arr insee_can          statut insee_com  \
0        27        70         2        05  Commune simple     70177   
1        27        89         1        13  Commune simple     89050   
2        32        60         4        13  Commune simple     60414  

In [7]:
import geopandas as gpd
import fiona

gpkg_path = r"D:\M2_MoSEF\Acclim\DataCollection\data\adminexpress-cog-simpl-000-2025.gpkg"
layer = "commune"

# 0) Confirm layer exists
layers = fiona.listlayers(gpkg_path)
print("Layers:", layers)
if layer not in layers:
    raise ValueError(f"Layer '{layer}' not found. Available: {layers}")

# 1) Read layer
gdf = gpd.read_file(gpkg_path, layer=layer)

print("\nLayer:", layer)
print("Rows:", len(gdf))
print("CRS:", gdf.crs)
print("Geometry types:", gdf.geom_type.value_counts().to_dict())

cols = list(gdf.columns)
print("Columns:", cols)

# 2) Auto-detect likely INSEE + name columns
def find_first(cols, keys):
    for c in cols:
        cu = c.upper()
        if any(k in cu for k in keys):
            return c
    return None

code_col = find_first(cols, ["INSEE", "CODE_INSEE", "CODGEO", "COM"])  # broad match
name_col = find_first(cols, ["NOM", "LIB", "NAME"])

print("\nDetected columns:")
print(" - code_col:", code_col)
print(" - name_col:", name_col)

# 3) Preview (if found)
preview_cols = [c for c in [code_col, name_col] if c is not None] + ["geometry"]
print("\nPreview:")
print(gdf[preview_cols].head(5))


Layers: ['commune_int', 'commune', 'departement', 'region', 'cheflieu', 'epci', 'departement_int', 'region_int', 'epci_int']

Layer: commune
Rows: 34877
CRS: EPSG:4326
Geometry types: {'Polygon': 34760, 'MultiPolygon': 117}
Columns: ['population', 'surf_adminexpress_geo_ha', 'id', 'siren_epci', 'insee_reg', 'insee_dep', 'insee_arr', 'insee_can', 'statut', 'insee_com', 'surf_adminexpress_prj_ha', 'nom', 'geometry']

Detected columns:
 - code_col: insee_reg
 - name_col: nom

Preview:
  insee_reg         nom                                           geometry
0        27       Corre  POLYGON ((5.96855 47.90474, 5.97163 47.91231, ...
1        27     Bonnard  POLYGON ((3.55654 47.92671, 3.54282 47.92768, ...
2        32  Montataire  POLYGON ((2.45536 49.24993, 2.45806 49.25314, ...
3        32    Thièvres  POLYGON ((2.46106 50.10407, 2.46523 50.10267, ...
4        28       Orbec  POLYGON ((0.37082 49.05014, 0.37002 49.04251, ...


In [9]:
# ============================================================
# Cell 1 — Imports + CONFIG (set your paths here)
# ============================================================
import os
import re
import glob
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.warp import transform_bounds
from rasterio.mask import mask
from shapely.geometry import box
from tqdm import tqdm

# --- PATHS (EDIT THESE) ---
FLOODS_ROOT = r"D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2024_filtered"  # folder containing year subfolders
COMMUNES_GPKG = r"D:\M2_MoSEF\Acclim\DataCollection\data\adminexpress-cog-simpl-000-2025.gpkg"
COMMUNES_LAYER = "commune"  # IMPORTANT: use polygon layer, not commune_int

OUT_WIDE = os.path.join(FLOODS_ROOT, "events_communes_wide.csv")
OUT_LONG = os.path.join(FLOODS_ROOT, "events_communes_long.csv")

# --- SEMANTICS ---
DEPTH_THRESHOLD_CM = 0.0            # commune affected if any pixel depth > this
IGNORE_VALUES = {0.0, 9999.0}       # 0=no flood, 9999=permanent/seasonal water

# Optional: strictness of "touch" on borders
ALL_TOUCHED = True

# Quick sanity check: list a few TIFs
tifs_preview = sorted(glob.glob(os.path.join(FLOODS_ROOT, "**", "*.tif"), recursive=True))[:5]
print("Preview TIFs:", *tifs_preview, sep="\n- ")
print("\nWill write:\n-", OUT_WIDE, "\n-", OUT_LONG)


Preview TIFs:
- D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2024_filtered\WD_MERGE_2024-01-01---2024-01-29_duration_28_days_cluster_335_A0_003215_A_003552_lat_05418_lon_01690_size_0452.tif
- D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2024_filtered\WD_MERGE_2024-01-01---2024-02-05_duration_35_days_cluster_333_A0_001861_A_002025_lat_04966_lon_00582_size_0555.tif
- D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2024_filtered\WD_MERGE_2024-01-01---2024-02-05_duration_35_days_cluster_336_A0_000603_A_000677_lat_04511_lon_02826_size_0106.tif
- D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2024_filtered\WD_MERGE_2024-01-01---2024-02-05_duration_35_days_cluster_337_A0_002307_A_002648_lat_06440_lon_02916_size_0209.tif
- D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2024_filtered\WD_MERGE_2024-01-01---2024-02-19_duration_35_days_cluster_334_A0_002595_A_002920_lat_05238_lon_01260_size_0503.tif

Will write:
- D:\M2_MoSEF\Accl

In [10]:
# ============================================================
# Cell 2 — Helpers (filename parsing + commune columns + test)
# ============================================================

# Filename pattern:
# WD_MERGE_[start]---[end]_duration_[d]_days_cluster_[id]_...
PAT = re.compile(
    r"WD_MERGE_(\d{4}-\d{2}-\d{2})---(\d{4}-\d{2}-\d{2})_duration_(\d+)_days_cluster_([^_]+)_.*\.tif$",
    re.IGNORECASE
)

def parse_event_from_name(path: str):
    name = os.path.basename(path)
    m = PAT.match(name)
    if not m:
        return None
    start_date, end_date, duration_days, cluster_id = m.groups()
    return {
        "event_file": name,
        "event_path": path,
        "start_date": start_date,
        "end_date": end_date,
        "duration_days": int(duration_days),
        "cluster_id": str(cluster_id),
    }

def pick_columns(communes: gpd.GeoDataFrame):
    """
    Your AdminExpress simplified gpkg has:
      - insee_com  (commune INSEE code)  ✅ use this
      - nom        (commune name)        ✅ use this
    """
    preferred_code = ["insee_com", "INSEE_COM", "codgeo", "CODGEO"]
    preferred_name = ["nom", "NOM", "nom_com", "NOM_COM", "libelle", "LIBELLE"]

    code_col = next((c for c in preferred_code if c in communes.columns), None)
    name_col = next((c for c in preferred_name if c in communes.columns), None)

    if code_col is None or name_col is None:
        raise ValueError(
            f"Couldn't find commune code/name columns. Available: {list(communes.columns)}"
        )
    return code_col, name_col

def commune_is_affected(src, geom, threshold_cm: float):
    # returns masked array when filled=False
    data, _ = mask(src, [geom], crop=True, filled=False, all_touched=ALL_TOUCHED, indexes=1)
    arr = data[0].astype(np.float32, copy=False)

    vals = arr.compressed() if np.ma.isMaskedArray(arr) else arr.ravel()
    if vals.size == 0:
        return False, None

    # filter ignore values
    for v in IGNORE_VALUES:
        vals = vals[vals != v]
    if vals.size == 0:
        return False, None

    max_cm = float(np.max(vals))
    return (max_cm > threshold_cm), max_cm


# --- Load communes + build spatial index (this will fail fast if anything is wrong) ---
communes = gpd.read_file(COMMUNES_GPKG, layer=COMMUNES_LAYER)
if communes.crs is None:
    raise ValueError("Communes layer has no CRS.")

code_col, name_col = pick_columns(communes)
print("Using columns:", code_col, name_col)

communes_wgs84 = communes.to_crs("EPSG:4326")
sindex = communes_wgs84.sindex  # requires rtree/pygeos; if it errors, install rtree

print("Communes loaded:", len(communes_wgs84))
print("CRS:", communes_wgs84.crs)
print(communes_wgs84[[code_col, name_col]].head())


Using columns: insee_com nom
Communes loaded: 34877
CRS: EPSG:4326
  insee_com         nom
0     70177       Corre
1     89050     Bonnard
2     60414  Montataire
3     80756    Thièvres
4     14478       Orbec


In [16]:
# ============================================================
# Cell 3 — FULL PIPELINE (build tables + save CSVs)
# ============================================================

tifs = sorted(glob.glob(os.path.join(FLOODS_ROOT, "**", "*.tif")))
print("Total TIFs found:", len(tifs))

long_rows = []
wide_rows = []
errors = []

for fp in tqdm(tifs, desc="Processing events"):
    meta = parse_event_from_name(fp)
    if meta is None:
        continue

    try:
        with rasterio.open(fp) as src:
            if src.crs is None:
                continue

            # Raster bounds in WGS84 for fast candidate filtering
            b = transform_bounds(src.crs, "EPSG:4326", *src.bounds, densify_pts=21)
            raster_bbox = box(*b)

            cand_idx = list(sindex.intersection(raster_bbox.bounds))
            if not cand_idx:
                wide_rows.append({**meta, "n_communes": 0, "communes": ""})
                continue

            cands = communes_wgs84.iloc[cand_idx]
            cands = cands[cands.intersects(raster_bbox)]
            if cands.empty:
                wide_rows.append({**meta, "n_communes": 0, "communes": ""})
                continue

            # Reproject candidates to raster CRS for masking
            cands_src = cands.to_crs(src.crs)

            affected = []
            for _, row in cands_src.iterrows():
                ok, max_cm = commune_is_affected(src, row.geometry, DEPTH_THRESHOLD_CM)
                if ok:
                    insee = row[code_col]
                    nom = row[name_col]
                    affected.append((insee, nom, max_cm))
                    long_rows.append({
                        **meta,
                        "insee_com": insee,
                        "commune": nom,
                        "max_depth_cm": max_cm,
                    })

            affected_sorted = sorted(affected, key=lambda x: (str(x[1]), str(x[0])))
            communes_str = "; ".join([f"{nom} ({insee})" for insee, nom, _ in affected_sorted])

            wide_rows.append({
                **meta,
                "n_communes": len(affected_sorted),
                "communes": communes_str,
            })

    except Exception as e:
        errors.append((fp, repr(e)))
        continue

df_wide = pd.DataFrame(wide_rows).sort_values(["start_date", "end_date", "cluster_id"])
df_long = pd.DataFrame(long_rows).sort_values(["start_date", "end_date", "cluster_id", "commune"])

df_wide.to_csv(OUT_WIDE, index=False, encoding="utf-8-sig")
df_long.to_csv(OUT_LONG, index=False, encoding="utf-8-sig")

print("\nSaved:")
print(" -", OUT_WIDE, "rows:", len(df_wide))
print(" -", OUT_LONG, "rows:", len(df_long))

# Show a quick preview in-notebook
display(df_wide.head(10))
display(df_long.head(10))

# Optional: review errors (if any)
if errors:
    print("\nErrors (first 10):")
    for fp, err in errors[:10]:
        print("-", os.path.basename(fp), "->", err)


Total TIFs found: 134


Processing events: 100%|██████████| 134/134 [11:08<00:00,  4.99s/it]


Saved:
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2024_filtered\events_communes_wide.csv rows: 134
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2024_filtered\events_communes_long.csv rows: 439


,event_file,event_path,start_date,end_date,duration_days,cluster_id,n_communes,communes
0,WD_MERGE_2024-01-01---2024-01-29_duration_28_d...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2024-01-01,2024-01-29,28,335,0,
1,WD_MERGE_2024-01-01---2024-02-05_duration_35_d...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2024-01-01,2024-02-05,35,333,130,Abaucourt (54001); Airel (50004); Ambly-Fleury...
2,WD_MERGE_2024-01-01---2024-02-05_duration_35_d...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2024-01-01,2024-02-05,35,336,0,
3,WD_MERGE_2024-01-01---2024-02-05_duration_35_d...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2024-01-01,2024-02-05,35,337,0,
4,WD_MERGE_2024-01-01---2024-02-19_duration_35_d...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2024-01-01,2024-02-19,35,334,9,Armaucourt (54021); Aulnois-sur-Seille (57040)...
5,WD_MERGE_2024-01-01---2024-04-01_duration_91_d...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2024-01-01,2024-04-01,91,332,2,Beuvrequen (62125); Wimille (62894)
6,WD_MERGE_2024-01-08---2024-01-15_duration_7_da...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2024-01-08,2024-01-15,7,003,0,
7,WD_MERGE_2024-01-08---2024-01-15_duration_7_da...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2024-01-08,2024-01-15,7,004,1,Plouider (29198)
8,WD_MERGE_2024-01-08---2024-01-15_duration_7_da...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2024-01-08,2024-01-15,7,005,0,
9,WD_MERGE_2024-01-08---2024-01-15_duration_7_da...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2024-01-08,2024-01-15,7,006,0,


,event_file,event_path,start_date,end_date,duration_days,cluster_id,insee_com,commune,max_depth_cm
83,WD_MERGE_2024-01-01---2024-02-05_duration_35_d...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2024-01-01,2024-02-05,35,333,54001,Abaucourt,17.0
32,WD_MERGE_2024-01-01---2024-02-05_duration_35_d...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2024-01-01,2024-02-05,35,333,50004,Airel,57.0
111,WD_MERGE_2024-01-01---2024-02-05_duration_35_d...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2024-01-01,2024-02-05,35,333,08010,Ambly-Fleury,19.0
43,WD_MERGE_2024-01-01---2024-02-05_duration_35_d...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2024-01-01,2024-02-05,35,333,50016,Appeville,69.0
104,WD_MERGE_2024-01-01---2024-02-05_duration_35_d...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2024-01-01,2024-02-05,35,333,51018,Athis,47.0
84,WD_MERGE_2024-01-01---2024-02-05_duration_35_d...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2024-01-01,2024-02-05,35,333,57040,Aulnois-sur-Seille,36.0
42,WD_MERGE_2024-01-01---2024-02-05_duration_35_d...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2024-01-01,2024-02-05,35,333,50023,Auvers,49.0
60,WD_MERGE_2024-01-01---2024-02-05_duration_35_d...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2024-01-01,2024-02-05,35,333,88031,Balléville,24.0
39,WD_MERGE_2024-01-01---2024-02-05_duration_35_d...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2024-01-01,2024-02-05,35,333,14038,Banville,16.0
21,WD_MERGE_2024-01-01---2024-02-05_duration_35_d...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2024-01-01,2024-02-05,35,333,61026,Barville,27.0


In [14]:
import rasterio, glob, os

one_tif = sorted(glob.glob(os.path.join(FLOODS_ROOT, "**", "*.tif"), recursive=True))[0]
with rasterio.open(one_tif) as src:
    print("CRS:", src.crs)
    print("Res:", src.res)  # should be ~ (20, 20) in metres

CRS: EPSG:27704
Res: (20.0, 20.0)
